# Experiment 8: LoRA during continued pretraining

Does restricting credit-domain CPT to low-rank attention updates change its training dynamics? The separate campaign has 48 planned runs: TabPFN v3 and TabICLv2, three learning rates, four held-out dataset partitions and PD/LGD. Rank 16 and alpha 32 stay fixed; only attention query/value LoRA factors are optimized. The target table remains excluded from CPT.

## 0. Evidence and scope

In [1]:
%matplotlib inline
import sys
from pathlib import Path
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from src.visualize import style, campaign as cp, lora_study as ls, publication as pub
from src.visualize.figures import FigureSaver
from src.visualize.inputs import source_description
style.apply()
sink = FigureSaver('experiment8/01_training')
report = cp.NotebookReport('Experiment 8: LoRA during continued pretraining', sink=sink)
print(source_description())
report.add('0. Evidence and scope', source_description() + '\n' + 'Training trajectories describe cross-table adaptation. Pending local evidence is not a live scheduler status; final prediction quality belongs in the results notebook.')
runs = [cp.load_campaign(8, track) for track in ('pd', 'lgd')]
from src.visualize import pilots as pv


DATA: no external source selected.
Project: no external source selected.
Missing project records are unavailable evidence, not zero effects or failed training.


## 1. LoRA recipe and training coverage

In [2]:
for run in runs:
    report.table(run.track.upper()+' protocol', ls.protocol_table(run))
    report.table(run.track.upper()+' evidence', pub.availability(run))
    report.table(run.track.upper()+' planned and recorded trials', ls.coverage_table(run))
report.add('1. LoRA recipe and training coverage', 'No measurements are synthesized for pending runs. This rank/alpha recipe is one constrained adaptation design; it does not establish an optimal LoRA configuration. Completed training does not imply completed benchmarking.')

,Setting,Value
0,Run identity,cpt_lora_v1
1,Task,PD
2,Adaptation,Low-rank updates to all attention query/value ...
3,Rank,16
4,Alpha,32
5,Adapter dropout,0.0
6,Learning rates,"1e-05, 0.0001, 0.001"
7,L2-SP,0
8,Planned trials in this task,24
9,Dataset partitions,4


,evidence,records,origin
0,Planned trials,24,Config; not scheduler state
1,Completed trials,0,DATA manifests
2,Epoch rows,0,Project training diagnostics
3,Milestone rows,0,Project trajectory diagnostics
4,Benchmark folds,0,Project final evaluation


,base,learning_rate,planned,completed,recorded_outcomes,evidence
0,TabICLv2,0.00001,4,0,PENDING: 4,No recorded training outcome
1,TabICLv2,0.00010,4,0,PENDING: 4,No recorded training outcome
2,TabICLv2,0.00100,4,0,PENDING: 4,No recorded training outcome
3,TabPFN v3,0.00001,4,0,PENDING: 4,No recorded training outcome
4,TabPFN v3,0.00010,4,0,PENDING: 4,No recorded training outcome
5,TabPFN v3,0.00100,4,0,PENDING: 4,No recorded training outcome


,Setting,Value
0,Run identity,cpt_lora_v1
1,Task,LGD
2,Adaptation,Low-rank updates to all attention query/value ...
3,Rank,16
4,Alpha,32
5,Adapter dropout,0.0
6,Learning rates,"1e-05, 0.0001, 0.001"
7,L2-SP,0
8,Planned trials in this task,24
9,Dataset partitions,4


,evidence,records,origin
0,Planned trials,24,Config; not scheduler state
1,Completed trials,0,DATA manifests
2,Epoch rows,0,Project training diagnostics
3,Milestone rows,0,Project trajectory diagnostics
4,Benchmark folds,0,Project final evaluation


,base,learning_rate,planned,completed,recorded_outcomes,evidence
0,TabICLv2,0.00001,4,0,PENDING: 4,No recorded training outcome
1,TabICLv2,0.00010,4,0,PENDING: 4,No recorded training outcome
2,TabICLv2,0.00100,4,0,PENDING: 4,No recorded training outcome
3,TabPFN v3,0.00001,4,0,PENDING: 4,No recorded training outcome
4,TabPFN v3,0.00010,4,0,PENDING: 4,No recorded training outcome
5,TabPFN v3,0.00100,4,0,PENDING: 4,No recorded training outcome


## 2. Loss, gradients and effective weight movement

One compact dashboard per model and task separates native query loss, optimizer gradients, effective attention-weight displacement and recorded training time. Curves require all four planned dataset partitions at each learning rate.

In [3]:
for run in runs:
    cp.show(sink, ls.plot_optimization(run))
    report.table(run.track.upper()+' numerical health', pv.health_table(run))
    for label, table in cp.loss_coverage_tables(run).items():
        report.table(run.track.upper()+' '+label, table)
report.add('2. Loss, gradients and effective weight movement', 'PD data loss is query cross-entropy. LGD retains the native objective of each architecture: TabPFN bar-distribution NLL or TabICL mean quantile pinball loss. No L2-SP term is applied. Effective Q/V displacement measures the induced weight update against initial targeted weights; it is not relative change from zero-valued adapter factors. Skipped and clipped updates remain in the health table, including exact zeros.')

No measurements available for this section yet.


""


""


No measurements available for this section yet.


""


""


## 3. Credit transfer and non-credit retention

Fixed-context AUC and R² monitors show changes during training. They are distinct from the five-fold evaluation of excluded tables.

In [4]:
for run in runs:
    cp.show(sink, ls.plot_monitors(run))
report.add('3. Credit transfer and non-credit retention', 'Each monitor observation is paired with the same table and trial at update zero. Tables receive equal weight; repeated non-credit partitions are averaged within table. Missing planned support leaves a gap rather than a zero effect.')

No measurements available for this section yet.
No measurements available for this section yet.


## 4. Learning-rate schedule and computational cost

In [5]:
cp.show(sink, ls.plot_schedule(runs))
for run in runs:
    report.table(run.track.upper()+' adapter parameters and training resources', ls.costs(run))
cp.show(sink, ls.plot_costs(runs))
report.add('4. Learning-rate schedule and computational cost', 'A shared configured schedule is shown once after training evidence exists; observed rates remain in its source table. The joint cost figure keeps task, model and rate separate. Completed-trial time excludes queue wait, and memory is peak allocated tensor memory. Missing costs remain unavailable; this is not an equal-compute comparison with full-parameter CPT.')

No measurements available for this section yet.


,trial_name,base,learning_rate,partition,seed,status
0,cpt_lora_v1_s00_pd_tabpfn-v3-classifier-v3_def...,TabPFN v3,0.00001,0,42,PENDING
1,cpt_lora_v1_s00_pd_tabpfn-v3-classifier-v3_def...,TabPFN v3,0.00010,0,42,PENDING
2,cpt_lora_v1_s00_pd_tabpfn-v3-classifier-v3_def...,TabPFN v3,0.00100,0,42,PENDING
3,cpt_lora_v1_s00_pd_tabicl-classifier-v2-202602...,TabICLv2,0.00001,0,42,PENDING
4,cpt_lora_v1_s00_pd_tabicl-classifier-v2-202602...,TabICLv2,0.00010,0,42,PENDING
5,cpt_lora_v1_s00_pd_tabicl-classifier-v2-202602...,TabICLv2,0.00100,0,42,PENDING
6,cpt_lora_v1_s01_pd_tabpfn-v3-classifier-v3_def...,TabPFN v3,0.00001,1,42,PENDING
7,cpt_lora_v1_s01_pd_tabpfn-v3-classifier-v3_def...,TabPFN v3,0.00010,1,42,PENDING
8,cpt_lora_v1_s01_pd_tabpfn-v3-classifier-v3_def...,TabPFN v3,0.00100,1,42,PENDING
9,cpt_lora_v1_s01_pd_tabicl-classifier-v2-202602...,TabICLv2,0.00001,1,42,PENDING


,trial_name,base,learning_rate,partition,seed,status
0,cpt_lora_v1_s00_lgd_tabpfn-v3-regressor-v3_def...,TabPFN v3,0.00001,0,42,PENDING
1,cpt_lora_v1_s00_lgd_tabpfn-v3-regressor-v3_def...,TabPFN v3,0.00010,0,42,PENDING
2,cpt_lora_v1_s00_lgd_tabpfn-v3-regressor-v3_def...,TabPFN v3,0.00100,0,42,PENDING
3,cpt_lora_v1_s00_lgd_tabicl-regressor-v2-202602...,TabICLv2,0.00001,0,42,PENDING
4,cpt_lora_v1_s00_lgd_tabicl-regressor-v2-202602...,TabICLv2,0.00010,0,42,PENDING
5,cpt_lora_v1_s00_lgd_tabicl-regressor-v2-202602...,TabICLv2,0.00100,0,42,PENDING
6,cpt_lora_v1_s01_lgd_tabpfn-v3-regressor-v3_def...,TabPFN v3,0.00001,1,42,PENDING
7,cpt_lora_v1_s01_lgd_tabpfn-v3-regressor-v3_def...,TabPFN v3,0.00010,1,42,PENDING
8,cpt_lora_v1_s01_lgd_tabpfn-v3-regressor-v3_def...,TabPFN v3,0.00100,1,42,PENDING
9,cpt_lora_v1_s01_lgd_tabicl-regressor-v2-202602...,TabICLv2,0.00001,1,42,PENDING


No measurements available for this section yet.


## Complete text summary

All displayed values, tables and captions in section order.

In [6]:
print(report.summary(sink))

Experiment 8: LoRA during continued pretraining

0. Evidence and scope
DATA: no external source selected.
Project: no external source selected.
Missing project records are unavailable evidence, not zero effects or failed training.
Training trajectories describe cross-table adaptation. Pending local evidence is not a live scheduler status; final prediction quality belongs in the results notebook.

1. LoRA recipe and training coverage
No measurements are synthesized for pending runs. This rank/alpha recipe is one constrained adaptation design; it does not establish an optimal LoRA configuration. Completed training does not imply completed benchmarking.

Table: PD protocol
CSV (first column: row index)
row,Setting,Value
0,Run identity,cpt_lora_v1
1,Task,PD
2,Adaptation,Low-rank updates to all attention query/value projections; base weights fixed
3,Rank,16
4,Alpha,32
5,Adapter dropout,0.0
6,Learning rates,"1e-05, 0.0001, 0.001"
7,L2-SP,0
8,Planned trials in this task,24
9,Dataset partition